# Exploration des Données : Élections Présidentielles 2022 — Tour 1 (Bureaux de vote, France entière)

Ce notebook explore les résultats du **1er tour** des élections présidentielles 2022 au niveau **bureau de vote** pour la **France entière**, filtré sur le département du Rhône (69).  
Source : [data.gouv.fr — resultats-par-niveau-burvot-t1-france-entiere.xlsx](https://static.data.gouv.fr/resources/election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx)  
Identifiant ressource : `98eb9dab-f328-4dee-ac08-ac17211357a8`  
Format : **XLSX** (31.9 Mo) → converti en **CSV** (`;` comme séparateur)


In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
import openpyxl

pd.set_option('display.max_rows', 500)


In [2]:


# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

XLSX_URL = (
    "https://static.data.gouv.fr/resources/"
    "election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/"
    "20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx"
)

path_xlsx_raw     = os.path.join(RAW_DIR, "2022_burvot_t1_france_entiere.xlsx")
path_rhone_bronze = os.path.join(BRONZE_DIR, "2022_bronze_burvot_t1_rhone_69.csv")



In [3]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_xlsx_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(XLSX_URL, timeout=180)
    resp.raise_for_status()
    with open(path_xlsx_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_xlsx_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


📥 Downloading source to RAW...
✅ Saved to RAW: ../../../data/raw/2022_burvot_t1_france_entiere.xlsx


In [4]:

# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_all = pd.read_excel(path_xlsx_raw, header=0, dtype=str, engine="openpyxl")

# Identify the department column
col_dept = next(c for c in df_all.columns if any(kw in c.lower() for kw in ["département", "dept"]))

# Filter for Rhône (69)
df_bronze = df_all[df_all[col_dept].astype(str).str.strip().str.lstrip("0") == "69"].copy()
print("____________________")
print(df_bronze.columns.to_list())

# --- RECONSTRUCT CANDIDATE COLUMNS (Structural Ingestion) ---
CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
cols = list(df_bronze.columns)
start_cand = next((i for i, c in enumerate(cols) if "panneau" in c.lower()), None)

if start_cand is not None:
    new_cols = cols[:start_cand]
    # Calculate candidates based on remaining original columns
    n_cand = (len(cols) - start_cand) // len(CAND_FIELDS)
    for k in range(n_cand):
        suffix = f".{k}" if k > 0 else ""
        for field in CAND_FIELDS:
            new_cols.append(f"{field}{suffix}")
    df_bronze.columns = new_cols

# --- ADD BRONZE METADATA ---
# Essential for the Medallion pattern to track data lineage
df_bronze['extraction_source_url'] = XLSX_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']     = os.path.basename(path_xlsx_raw)

# ── 4. SAVE TO BRONZE ───────────────────────────────────────────────────────
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")
display(df_bronze.head())

print(f"✅ BRONZE dataset created: {path_rhone_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


🛠 Processing RAW -> BRONZE...
____________________
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnam

,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,...,N°Panneau.11,Sexe.11,Nom.11,Prénom.11,Voix.11,% Voix/Ins.11,% Voix/Exp.11,extraction_source_url,ingestion_timestamp,source_file_name
47273,69,Rhône,08,8ème circonscription,001,Affoux,0001,316,60,18.99,...,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47274,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,217,58,26.73,...,12,M,DUPONT-AIGNAN,Nicolas,7,3.23,4.58,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47275,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,999,194,19.42,...,12,M,DUPONT-AIGNAN,Nicolas,16,1.6,2.03,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47276,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,862,224,25.99,...,12,M,DUPONT-AIGNAN,Nicolas,6,0.7,0.96,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47277,69,Rhône,09,9ème circonscription,004,Alix,0001,470,73,15.53,...,12,M,DUPONT-AIGNAN,Nicolas,10,2.13,2.57,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx


✅ BRONZE dataset created: ../../../data/bronze/2022_bronze_burvot_t1_rhone_69.csv
📊 Rows: 1317 | Columns: 108


In [5]:
df_bronze.dtypes

pd.set_option('display.max_columns', None)

display(df_bronze.head())


,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,N°Panneau.1,Sexe.1,Nom.1,Prénom.1,Voix.1,% Voix/Ins.1,% Voix/Exp.1,N°Panneau.2,Sexe.2,Nom.2,Prénom.2,Voix.2,% Voix/Ins.2,% Voix/Exp.2,N°Panneau.3,Sexe.3,Nom.3,Prénom.3,Voix.3,% Voix/Ins.3,% Voix/Exp.3,N°Panneau.4,Sexe.4,Nom.4,Prénom.4,Voix.4,% Voix/Ins.4,% Voix/Exp.4,N°Panneau.5,Sexe.5,Nom.5,Prénom.5,Voix.5,% Voix/Ins.5,% Voix/Exp.5,N°Panneau.6,Sexe.6,Nom.6,Prénom.6,Voix.6,% Voix/Ins.6,% Voix/Exp.6,N°Panneau.7,Sexe.7,Nom.7,Prénom.7,Voix.7,% Voix/Ins.7,% Voix/Exp.7,N°Panneau.8,Sexe.8,Nom.8,Prénom.8,Voix.8,% Voix/Ins.8,% Voix/Exp.8,N°Panneau.9,Sexe.9,Nom.9,Prénom.9,Voix.9,% Voix/Ins.9,% Voix/Exp.9,N°Panneau.10,Sexe.10,Nom.10,Prénom.10,Voix.10,% Voix/Ins.10,% Voix/Exp.10,N°Panneau.11,Sexe.11,Nom.11,Prénom.11,Voix.11,% Voix/Ins.11,% Voix/Exp.11,extraction_source_url,ingestion_timestamp,source_file_name
47273,69,Rhône,08,8ème circonscription,001,Affoux,0001,316,60,18.99,256,81.01,4,1.27,1.56,2,0.63,0.78,250,79.11,97.66,1,F,ARTHAUD,Nathalie,0,0,0,2,M,ROUSSEL,Fabien,4,1.27,1.6,3,M,MACRON,Emmanuel,58,18.35,23.2,4,M,LASSALLE,Jean,17,5.38,6.8,5,F,LE PEN,Marine,88,27.85,35.2,6,M,ZEMMOUR,Éric,22,6.96,8.8,7,M,MÉLENCHON,Jean-Luc,27,8.54,10.8,8,F,HIDALGO,Anne,0,0,0,9,M,JADOT,Yannick,6,1.9,2.4,10,F,PÉCRESSE,Valérie,17,5.38,6.8,11,M,POUTOU,Philippe,0,0,0,12,M,DUPONT-AIGNAN,Nicolas,11,3.48,4.4,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47274,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,217,58,26.73,159,73.27,5,2.3,3.14,1,0.46,0.63,153,70.51,96.23,1,F,ARTHAUD,Nathalie,1,0.46,0.65,2,M,ROUSSEL,Fabien,2,0.92,1.31,3,M,MACRON,Emmanuel,48,22.12,31.37,4,M,LASSALLE,Jean,8,3.69,5.23,5,F,LE PEN,Marine,48,22.12,31.37,6,M,ZEMMOUR,Éric,9,4.15,5.88,7,M,MÉLENCHON,Jean-Luc,9,4.15,5.88,8,F,HIDALGO,Anne,2,0.92,1.31,9,M,JADOT,Yannick,2,0.92,1.31,10,F,PÉCRESSE,Valérie,15,6.91,9.8,11,M,POUTOU,Philippe,2,0.92,1.31,12,M,DUPONT-AIGNAN,Nicolas,7,3.23,4.58,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47275,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,999,194,19.42,805,80.58,11,1.1,1.37,6,0.6,0.75,788,78.88,97.89,1,F,ARTHAUD,Nathalie,2,0.2,0.25,2,M,ROUSSEL,Fabien,9,0.9,1.14,3,M,MACRON,Emmanuel,263,26.33,33.38,4,M,LASSALLE,Jean,20,2,2.54,5,F,LE PEN,Marine,124,12.41,15.74,6,M,ZEMMOUR,Éric,70,7.01,8.88,7,M,MÉLENCHON,Jean-Luc,156,15.62,19.8,8,F,HIDALGO,Anne,14,1.4,1.78,9,M,JADOT,Yannick,55,5.51,6.98,10,F,PÉCRESSE,Valérie,55,5.51,6.98,11,M,POUTOU,Philippe,4,0.4,0.51,12,M,DUPONT-AIGNAN,Nicolas,16,1.6,2.03,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47276,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,862,224,25.99,638,74.01,6,0.7,0.94,4,0.46,0.63,628,72.85,98.43,1,F,ARTHAUD,Nathalie,2,0.23,0.32,2,M,ROUSSEL,Fabien,6,0.7,0.96,3,M,MACRON,Emmanuel,197,22.85,31.37,4,M,LASSALLE,Jean,21,2.44,3.34,5,F,LE PEN,Marine,154,17.87,24.52,6,M,ZEMMOUR,Éric,51,5.92,8.12,7,M,MÉLENCHON,Jean-Luc,115,13.34,18.31,8,F,HIDALGO,Anne,8,0.93,1.27,9,M,JADOT,Yannick,32,3.71,5.1,10,F,PÉCRESSE,Valérie,33,3.83,5.25,11,M,POUTOU,Philippe,3,0.35,0.48,12,M,DUPONT-AIGNAN,Nicolas,6,0.7,0.96,https://static.data.gouv.fr/resources/election...,2026-03-18T14:29:03.676545,2022_burvot_t1_france_entiere.xlsx
47277,69,Rhône,09,9ème circonscription,004,Alix,0001,470,73,15.53,397,84.47,6,1.28,1.51,2,0.43,0.5,389,82.77,97.98,1,F,ARTHAUD,Nathalie,1,0.21,0.26,2,M,ROUSSEL,Fabien,11,2.34,2.83,3,M,MACRON,Emmanuel,112,23.83,28.79,4,M,LASSALLE,Jean,8,1.7,2.06,5,F,LE PEN,Marine,102,21.7,26.22,6,M,ZEMMOUR,Éric,36,7.66,9.25,7,M,MÉLENCHON,Jean-Luc,53,11.28,13.62